# CPA attack on standard ASCON software implementation

This notebook performs the **CPA attack** on the previously acquired power traces.  
It loads the traceset produced by the acquisition notebook ( the h5 file (.h5)) and computes:

- **Correlation Power Analysis (CPA)** results  
- **Key rank** vs. number of traces  
- **Correlation trends** for each key byte, comparing correct vs. wrong key hypotheses  

These plots help evaluate the **difficulty of recovering each key byte** and visualize the leakage behavior across the trace window.

⚠️ **Important:**  
This notebook assumes that the **Power Trace Acquisition** notebook (`xheep_capture_ASCON.ipynb`) has already been executed and the traceset has been acquired.


In [1]:
sbox_type   = "lut_ascon"   # do not modify this line
tested_sbox = sbox_type     # alias used later in the printout
n_trc       = 10_000        # total number of traces in the traceset

In [2]:
import sys
import os
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import logging
import json
import h5py

## Project paths
This block robustly detects the project root (`DOJO_ROOT`) starting from either the script location (`__file__`) or the current working directory (for notebooks). From there it defines all relevant subdirectories (ASCON sources, SCA scripts, X-HEEP, traces, plots, cache), ensures output folders exist, and adds the local source paths to `sys.path` .

In [3]:
def find_project_root(start: Path, markers=("fusesoc.conf", ".dojo_root")) -> Path:
    current = start
    while current != current.parent:
        if any((current / m).exists() for m in markers):
            return current
        current = current.parent

    raise RuntimeError(
        f"Could not find project root (looked for markers: {markers}). "
        "Please ensure you are inside the Side-Channel-Dojo repository."
    )

# In a script, __file__ exists; in a notebook it does not.
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()

DOJO_ROOT = find_project_root(SCRIPT_DIR)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

ASCON_PY_DIR = DOJO_ROOT / "sw" / "ciphers" / "ASCON_init_python"
SCA_DIR      = DOJO_ROOT / "sw" / "sca_scripts"
HW_DIR       = DOJO_ROOT / "hw"

# Base dirs for ASCON SW SCA
BASE_PLOT_DIR  = DOJO_ROOT / "sw" / "sca_scripts" / "ASCON" / "sw" / "plot"
BASE_CACHE_DIR = DOJO_ROOT / "sw" / "sca_scripts" / "ASCON" / "sw" / "cache"

# Traceset (HDF5) for ASCON SW
TRACESET_DIR  = DOJO_ROOT / "sw" / "traceset" / "ASCON" / "sw"
TRACESET_FILE = TRACESET_DIR / f"ascon_opt32_{sbox_type}_{n_trc // 1000}k.h5"

# Per-S-box plot and cache dirs
PLOT_DIR       = BASE_PLOT_DIR / sbox_type
CPA_CACHE_FILE = BASE_CACHE_DIR / sbox_type / f"CPA_results_{sbox_type}.json"

# Ensure directories exist
BASE_PLOT_DIR.mkdir(parents=True, exist_ok=True)
BASE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
CPA_CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)

# Import local modules
sys.path.insert(0, str(ASCON_PY_DIR))
sys.path.insert(0, str(SCA_DIR))

## Imports

In [4]:
from analyzer.attack.ascon.xheep_ascon_cpa.ascon_first_round import ascon_first_round
from analyzer.attack.ascon.xheep_ascon_cpa.ascon_std_leakage_model import ascon_std_leakage_model
from analyzer.attack.ascon.xheep_ascon_cpa.ascon_cpa import ascon_cpa

# Configuration

In [5]:
# ---------------------------------------------------------------------------
# Flow flags
# ---------------------------------------------------------------------------
attack_single_bit        = False  # Attack a single key bit (for testing / debugging)
verbose                  = True   # Verbose output during key recovery

traces_overlapped_plot   = True   # Plot overlapped power traces

# CPA / analysis cache control
load_attack_results      = False  # Load CPA cache if available
save_attack_results      = True   # Save CPA results to cache after the run

# Plot control
key_rank_plot            = True   # Plot PGE vs traces
traces_correlation_plot  = True   # Plot correlation vs traces

# Output control
save_plots               = True   # Save plots to disk
save_results             = True   # Save analysis results (JSON, etc.) to disk

# ---------------------------------------------------------------------------
# Configuration printout
# ---------------------------------------------------------------------------

def _yn(flag: bool) -> str:
    """Return 'yes' or 'no' for a boolean flag."""
    return "yes" if flag else "no"

print("\n================= CONFIGURATION =================")
print(f"DOJO_ROOT           : {DOJO_ROOT}")
print()
print("Target")
print(f"  Cipher                    : ASCON (standard permutation)")
print(f"  S-box implementation      : {tested_sbox}  [STANDARD ASCON S-box]")
print(f"  Notebook scope            : ASCON SW, standard S-box only")
print()
print("Paths")
print(f"  Traceset file             : {TRACESET_FILE}")
print(f"  Plot dir                  : {PLOT_DIR}")
print(f"  CPA cache file            : {CPA_CACHE_FILE}")
print()
print("Analysis configuration")
print(f"  Total traces (n_trc)      : {n_trc}")
print(f"  Save plots                : {_yn(save_plots)}")
print(f"  Save results              : {_yn(save_results)}")
print(f"  Load CPA results (cache)  : {_yn(load_attack_results)}")
print(f"  Save CPA results (cache)  : {_yn(save_attack_results)}")
print()
print(f"  Key rank plot             : {_yn(key_rank_plot)}")
print(f"  Correlation plot          : {_yn(traces_correlation_plot)}")
print("=================================================\n")


================= CONFIGURATION =================
DOJO_ROOT           : /home/mattia-mirigaldi/Desktop/Side-Channel-Dojo

Target
  Cipher                    : ASCON (standard permutation)
  S-box implementation      : lut_ascon  [STANDARD ASCON S-box]
  Notebook scope            : ASCON SW, standard S-box only

Paths
  Traceset file             : /home/mattia-mirigaldi/Desktop/Side-Channel-Dojo/sw/traceset/ASCON/sw/ascon_opt32_lut_ascon_10k.h5
  Plot dir                  : /home/mattia-mirigaldi/Desktop/Side-Channel-Dojo/sw/sca_scripts/ASCON/sw/plot/lut_ascon
  CPA cache file            : /home/mattia-mirigaldi/Desktop/Side-Channel-Dojo/sw/sca_scripts/ASCON/sw/cache/lut_ascon/CPA_results_lut_ascon.json

Analysis configuration
  Total traces (n_trc)      : 10000
  Save plots                : yes
  Save results              : yes
  Load CPA results (cache)  : no
  Save CPA results (cache)  : yes

  Key rank plot             : yes
  Correlation plot          : yes



## SCA attack

## Load data to perform the attack
- Opens the HDF5 traces file `TRACESET_FILE` in read mode.
- Checks that both datasets `"traces"` and `"nonces"` are present.
- Reads:
  - Up to `n_trc` traces and their corresponding nonces.
  - Metadata: `sampling_interval`, `n_samples`, `key`, `iv`.
- Verifies that the number of traces matches the number of nonces.
- Warns if the requested `n_trc` differs from the number of traces stored.


In [6]:
try:
    with h5py.File(TRACESET_FILE, "r") as f_read_traces:
        # Basic sanity: check that required datasets exist
        if "traces" not in f_read_traces or "nonces" not in f_read_traces:
            raise KeyError(
                "HDF5 file is missing required datasets 'traces' and/or 'nonces'."
            )

        traces_ds = f_read_traces["traces"]
        nonces_ds = f_read_traces["nonces"]

        total_traces = traces_ds.shape[0]
        n_samples    = traces_ds.shape[1]

        # Load metadata attributes (if present)
        sampling_interval   = f_read_traces.attrs.get("sampling_interval", None)
        n_samples           = f_read_traces.attrs.get("n_samples", n_samples)
        key                 = f_read_traces.attrs.get("key_hex", None) 
        iv                  = f_read_traces.attrs.get("iv_hex", None)

        # Use at most n_trc traces, but do not exceed what's in the file
        n_used = min(n_trc, total_traces)

        # Use slicing so data are actually loaded into RAM
        traces = traces_ds[:n_used]
        nonces = nonces_ds[:n_used]

    # Sanity check: traces and nonces should have the same number of rows
    if traces.shape[0] != nonces.shape[0]:
        raise ValueError(
            f"Number of traces ({traces.shape[0]}) and nonces ({nonces.shape[0]}) "
            "do not match. Check the traces file."
        )

    # Optional sanity checks vs metadata
    if n_trc != total_traces:
        print(
            f"[WARN] Wanted n_trc={n_trc} "
            f"differs from dataset length={total_traces}"
        )
        
        
    nonce_msb = int(nonces[0, 0])  # 0x0F0E0D0C0B0A0908
    nonce_lsb = int(nonces[0, 1])  # 0x0706050403020100
    # Rebuild the 128-bit value as stored (byte-reversed version)
    nonce_stored_int = (nonce_msb << 64) | nonce_lsb
    # Undo the byte reversal to recover the original big-endian nonce
    nonce_original_bytes = nonce_stored_int.to_bytes(16, byteorder="big")[::-1]
    nonce_original_hex = nonce_original_bytes.hex().upper()

    nonce_ini_hex = nonces[0]
    print(f"[INFO] Loaded {traces.shape[0]} traces from {TRACESET_FILE}")
    print(f"[INFO] Sampling interval      : {sampling_interval}")
    print(f"[INFO] Samples per trace      : {n_samples}")
    print(f"[INFO] Key                    : 0x{key}")
    print(f"[INFO] IV                     : 0x{iv}")
    print(f"[INFO] Initial nonce          : 0x{nonce_original_hex}")

except FileNotFoundError:
    print(
        f"[ERROR] Traces file {TRACESET_FILE} not found. "
        "Please run the trace acquisition phase first."
    )
except Exception as e:
    print(f"[ERROR] Could not read traces file {TRACESET_FILE}: {e}")


[INFO] Loaded 10000 traces from /home/mattia-mirigaldi/Desktop/Side-Channel-Dojo/sw/traceset/ASCON/sw/ascon_opt32_lut_ascon_10k.h5
[INFO] Sampling interval      : 8e-09
[INFO] Samples per trace      : 1125
[INFO] Key                    : 0x000102030405060708090A0B0C0D0E0F
[INFO] IV                     : 0x00001000808C0001
[INFO] Initial nonce          : 0x000102030405060708090A0B0C0D0E0F


### Single-bit CPA attack on standard ASCON S-box

In the standard ASCON S-box, each output bit can be written in Algebraic Normal Form (ANF), i.e., as a Boolean polynomial over $GF(2)$.  
For power/EM analysis we are interested **only in nonlinear key–nonce products**, because they modulate the switching activity in a key-dependent way.

From the ANF, and grouping $x_1$ and $x_2$ as an indistinguishable term $x_{12} = x_1 \oplus x_2$, we rename:
- $x_1 \to k''$ (first half of the key)
- $x_{12} \to k'$ (combined key term)
- $x_3 \to m''$, $x_4 \to m'$ (nonce halves)

The useful S-box outputs are the ones containing nonlinear mixes like $m'' \cdot k''$ or $m'' \cdot k'$; purely nonce-only terms (e.g. $m' \cdot (m'' + 1)$) are **not** exploitable for key recovery.

A crucial observation from the ANF is that, for the chosen attacked bit of $x_0$ after the first round, the **MSB half of the key does not appear** in the nonlinear term.  
This enables a **two-phase attack**:

1. **Phase 1 – Attack $x_0$ (recover 3 bits of the first key half)**  
   - Target a bit whose expression contains a term like $m'' \cdot k''$.  
   - Only 3 key bits influence this bit in a nonlinear way, so we treat **exactly 3 bits of $k''$ as unknown** and keep the rest as constants.  
   - By CPA on that bit, we recover those 3 key bits.

2. **Phase 2 – Attack $x_1$ (recover 3 bits of the second key half)**  
   - The 3 bits recovered from $x_0$ are now treated as known.  
   - We move to a bit in $x_1$ whose ANF contains a different triple of key bits (second half).  
   - Again, only 3 bits are unknown, leading to another 8 hypotheses and a second CPA step to recover them.

---

### Leakage model construction (standard S-box, 3-bit model)

For a **single attacked bit** after the first-round substitution + diffusion:

- At any time, we assume **only 3 key bits are unknown**.
- This gives **8 hypotheses**: all 3-bit values $k \in \{0, \dots, 7\}$.

For each trace (indexed by $n$):

1. Extract the nonce halves:
   - `nonce_MSB = nonces[n][1]`  
   - `nonce_LSB = nonces[n][0]`
2. Call  
   `leakage_model = ascon_leakage_model(init_vect, nonce_MSB, nonce_LSB, state_register_index, bit_index, sbox_type, key_0=...)`
3. This returns an array of length 8:
   - `leakage_model[k]` is the predicted value (0/1) of the attacked bit for key guess $k$.

Stacking this over $N$ traces yields the hypothetical leakage matrix:
- `H_matrix` of shape $(N, 8)$:
  - rows: traces / nonces
  - columns: 3-bit key guesses $k = 0..7$

This is the **standard-ASCON leakage model**: a single-bit selection function mapping each nonce and 3-bit key guess to a predicted output bit.

---

### CPA step on a single bit

Given:
- `traces` of shape $(N, M)$ (N traces, M samples per trace)
- `H_matrix` of shape $(N, 8)$

the function `ascon_cpa(traces, H_matrix)`:

1. For each key guess $k$ and each time sample $t$:
   - `x = traces[:, t]` → measured power at time $t$  
   - `y = H_matrix[:, k]` → predicted leakage for guess $k$  
   - compute Pearson correlation:
     - $R[k, t] = \mathrm{corr}(x, y)$

2. This yields `R_matrix` of shape $(8, M)$.

3. To score each key guess:
   - `corr_vs_keyguess = np.max(np.abs(R_matrix), axis=1)`  
   - one value per 3-bit hypothesis (max absolute correlation across time samples)

As the number of traces grows, the **correct 3-bit key guess** should exhibit the **largest correlation**, allowing us to recover 3 bits of the key from a single attacked state bit in the first round.


In [7]:
# Reverse key by bytes (2 hex chars per byte) to match C endianness
key_bytes       = [key[i:i+2] for i in range(0, len(key), 2)]
key_reversed    = "".join(key_bytes[::-1])
key_int         = int(key_reversed, 16)

# IV as integer (already in correct endianness)
iv_int = int(iv, 16)

In [8]:
if attack_single_bit:
    # -------------------------------------------------------------------
    # Single-bit CPA attack configuration
    # -------------------------------------------------------------------
    verbose = False  # enable detailed step-by-step output
    
    # -------------------------------------------------------------------
    # Basic attack configuration
    # -------------------------------------------------------------------
    state_register_index = 0      # attack x0
    bit_index            = 32     # attacked bit index in x0
    resolution           = 50   # traces per CPA step
    
    # First half of the key (64 bits), used in leakage model
    key_0 = key_int & 0xFFFFFFFFFFFFFFFF
    
    # DEBUG: extract the 3 correct key bits expected for this bit_index
    key_0_j   = (key_0 >> (bit_index % 64)) & 1
    key_0_j19 = (key_0 >> ((bit_index + 19) % 64)) & 1
    key_0_j28 = (key_0 >> ((bit_index + 28) % 64)) & 1
    
    print("=== Single-bit CPA configuration (standard ASCON) ===")
    print(f"  State register index         : {state_register_index}")
    print(f"  Attacked bit index           : {bit_index}")
    print(f"  True key bits (j, j+19, j+28): "
          f"({key_0_j}, {key_0_j19}, {key_0_j28})")
    
    corr_vs_traces = []  # will store correlation for all 8 key guesses at each step
    
    tic = time.perf_counter()
    
    # Number of CPA steps (each uses 'resolution' traces)
    n_steps = traces.shape[0] // resolution
    
    for step in range(1, n_steps + 1):
        n_used = step * resolution
    
        partial_traces = traces[:n_used]
        partial_nonces = nonces[:n_used]
    
        # Build the leakage model matrix for all nonces: shape (n_used, 8)
        H_matrix = np.empty((n_used, 8), dtype=np.uint8)
    
        for i, (nonce_lsb, nonce_msb) in enumerate(partial_nonces):
            leakage_model_i = ascon_std_leakage_model(
                iv_int,
                int(nonce_msb),  # MSB 64 bits
                int(nonce_lsb),  # LSB 64 bits
                state_register_index,
                bit_index,
                sbox_type,
            )
            H_matrix[i] = leakage_model_i
    
        # CPA attack: correlations for all 8 key hypotheses
        R_matrix = ascon_cpa(partial_traces, H_matrix)
    
        # For each key guess, take maximum absolute correlation over time samples
        corr_vs_keyguess = np.max(np.abs(R_matrix), axis=1)  # shape (8,)
        if verbose:
            print(f"\n[STEP {step}/{n_steps}] Number of traces: {n_used}")
            for j, max_corr_value in enumerate(corr_vs_keyguess):
                print(f"  Key guess {j}: {max_corr_value:.4f}")
    
            # Best key guess at this step
            best_key_guess = int(np.argmax(corr_vs_keyguess))
            print(f"  Best key guess        : {best_key_guess}")
            print(
                "  Expected key bits     : "
                f"({key_0_j}, {key_0_j19}, {key_0_j28}), got: "
                f"({(best_key_guess >> 2) & 1}, "
                f"{(best_key_guess >> 1) & 1}, "
                f"{best_key_guess & 1})"
            )
    
        # Store correlation values for all 8 guesses at this trace count
        corr_vs_traces.append(corr_vs_keyguess)
    
    # Determine the best key guess from the attack
    best_key_guess = int(np.argmax(corr_vs_keyguess))
    if (key_0_j, key_0_j19, key_0_j28) == ((best_key_guess >> 2) & 1, (best_key_guess >> 1) & 1,best_key_guess & 1,):
        print(f"[INFO] Correct key bits recovered. ")
    else :
        print(f"[INFO] Incorrect key bits recovered, best guess: {best_key_guess}.")
    
    toc = time.perf_counter()
    print(f"\nCPA attack on bit {bit_index} completed in {(toc - tic)/60:.2f} minutes.\n")

# Correlation vs number of traces plot

In [9]:
if attack_single_bit and traces_correlation_plot:
    # -------------------------------------------------------------------
    # Plot: correlation vs number of traces for all key guesses
    # -------------------------------------------------------------------
    corr_vs_traces = np.array(corr_vs_traces)  # shape (steps, 8)
    x = np.arange(1, len(corr_vs_traces) + 1) * resolution
    plt.figure(figsize=(10, 5))
    for key_idx in range(8):
        plt.plot(x, corr_vs_traces[:, key_idx], label=f"Key guess {key_idx}")
    plt.xlabel("Number of traces")
    plt.ylabel("Maximum absolute correlation")
    plt.title(f"Correlation vs number of traces - bit {bit_index} - S-box {sbox_type}")
    plt.grid(True, alpha=0.3, linestyle="--")
    plt.legend()
    plt.show()
    plt.close()

# Recovering the full key
TO recover the full key first the key index are taken by running the SNR to select which among the register x0 or x1 bits has the highest value and can lead to better attack

In [10]:
# questi devono essere presi da un cache file 
key_bit_indexes_0 = [32, 13, 34, 4, 6, 54, 36, 0, 33, 63, 7, 16, 55, 19, 17, 41, 1, 40, 8, 48, 24, 39, 14, 31, 58, 49, 56, 47, 37, 29, 15, 46, 57, 11]
key_bit_indexes_1 = [32, 0, 63, 1, 14, 13, 36, 15, 31, 8, 38, 43, 5, 18, 23, 12, 45, 16, 9, 42, 3, 51, 2, 49, 24, 20, 44, 40, 28, 30, 37, 19, 47, 59, 53, 4, 46]

In [11]:
k0_bits = np.zeros(64, dtype=np.uint8)
k0_recovered = np.zeros(64, dtype=bool)
k1_bits = np.zeros(64, dtype=np.uint8)
k1_recovered = np.zeros(64, dtype=bool)

def build_H_matrix(state_register_index: int,
                   bit_index: int,
                   nonces: np.ndarray,
                   initialization_vector: int,
                   sbox_type: str,
                   k0=None) -> np.ndarray:
    """
    Build the hypothetical leakage matrix H for a given state register and bit index.
    H has shape (n_traces, 8), one column per each of the 3-bit key hypothesis.
    """
    n_traces = len(nonces)
    H = np.empty((n_traces, 8), dtype=np.uint8)

    for i, (nonce_lsb, nonce_msb) in enumerate(nonces):
        if state_register_index == 0:
            leakage_model_i = ascon_std_leakage_model(
                initialization_vector,
                int(nonce_msb),
                int(nonce_lsb),
                0,
                bit_index,
                sbox_type,
            )
        else:
            # For state register x1, k0 (most significant half) is already recovered
            leakage_model_i = ascon_std_leakage_model(
                initialization_vector,
                int(nonce_msb),
                int(nonce_lsb),
                1,
                bit_index,
                sbox_type,
                k0,
            )
        H[i] = leakage_model_i

    return H


tic = time.perf_counter()
print()

# -------------------------------------------------------------------
# Recover k0 (most significant 64 bits)
# -------------------------------------------------------------------
print("=== Recovering k0 (most significant 64 bits) ===")
for key_bit in tqdm(key_bit_indexes_0, desc="Key bit recovery (k0)"): 
    # Build leakage model for this bit position
    H_matrix = build_H_matrix(
        state_register_index=0,
        bit_index=key_bit,
        nonces=nonces,
        initialization_vector=iv_int,
        sbox_type=sbox_type,
    )

    # CPA attack
    R_matrix = ascon_cpa(traces, H_matrix)

    # Max absolute correlation per key guess (shape (8,))
    corr_vs_keyguess = np.max(np.abs(R_matrix), axis=1)

    # Best key guess (3 bits)
    best_key_guess = int(np.argmax(corr_vs_keyguess))

    idx0  = key_bit % 64
    idx19 = (key_bit + 19) % 64
    idx28 = (key_bit + 28) % 64

    val0  = (best_key_guess >> 2) & 1
    val19 = (best_key_guess >> 1) & 1
    val28 = (best_key_guess >> 0) & 1

    for idx, val in ((idx0, val0), (idx19, val19), (idx28, val28)):
        if not k0_recovered[idx]:
            # First time we recover this bit → store it
            k0_bits[idx] = val
            k0_recovered[idx] = True
        elif k0_bits[idx] != val:
            # Already recovered with a different value → keep old one & warn
             if verbose: print(f"[WARN] Conflicting recovery for k0 bit {idx}: "
                  f"existing={k0_bits[idx]}, new={val} (ignored)")
        
# Build k0 from bit array (LSB at index 0)
k0 = 0
for i, bit in enumerate(k0_bits):
    k0 |= (int(bit) & 0x01) << i
k0 = k0 & 0xFFFFFFFFFFFFFFFF

print(f"Recovered most significant half of the key (k0): {k0:016X}\n")

# -------------------------------------------------------------------
# Recover k1 (least significant 64 bits)
# -------------------------------------------------------------------
print("=== Recovering k1 (least significant 64 bits) ===")
for key_bit in tqdm(key_bit_indexes_1, desc="Key bit recovery (k1)"):
    # Build leakage model for this bit position, now with k0 known
    H_matrix = build_H_matrix(
        state_register_index=1,
        bit_index=key_bit,
        nonces=nonces,
        initialization_vector=iv_int,
        sbox_type=sbox_type,
        k0=k0,
    )

    # CPA attack
    R_matrix = ascon_cpa(traces, H_matrix)

    # Max absolute correlation per key guess (shape (8,))
    corr_vs_keyguess = np.max(np.abs(R_matrix), axis=1)

    # Best key guess (3 bits)
    best_key_guess = int(np.argmax(corr_vs_keyguess))

    # later, in the k1 loop:
    idx0  = key_bit % 64
    idx61 = (key_bit + 61) % 64
    idx39 = (key_bit + 39) % 64

    val0  = (best_key_guess >> 2) & 1
    val61 = (best_key_guess >> 1) & 1
    val39 = (best_key_guess >> 0) & 1

    for idx, val in ((idx0, val0), (idx61, val61), (idx39, val39)):
        if not k1_recovered[idx]:
            k1_bits[idx] = val
            k1_recovered[idx] = True
        elif k1_bits[idx] != val:
            if verbose: print(f"[WARN] Conflicting recovery for k1 bit {idx}: "
                  f"existing={k1_bits[idx]}, new={val} (ignored)")

# Build k1 from bit array (LSB at index 0)
k1 = 0
for i, bit in enumerate(k1_bits):
    k1 |= (int(bit) & 0x01) << i
k1 = k1 & 0xFFFFFFFFFFFFFFFF

# The key recovery from the register z1 actually need this extra XOR operation
# k1 = k1 ^ k0

print(f"Recovered least significant half of the key (k1): {k1:016X}\n")
print(f"Recovered full key (k1||k0): {k1:016X}{k0:016X}")

# -------------------------------------------------------------------
# Pretty-print key in little-endian hex (for comparison)
# -------------------------------------------------------------------
k0_hex = f"{k0:016X}"
k1_hex = f"{k1:016X}"\
    
# Full key in big-endian hex (k1 MSB || k0 LSB)
recovered_key_be = k1_hex + k0_hex

# Convert to little-endian byte order (as in your firmware)
recovered_key_bytes = [
    recovered_key_be[i:i + 2] for i in range(0, len(recovered_key_be), 2)][::-1]
recovered_key_le = "".join(recovered_key_bytes)

print(f"Recovered key : 0x{recovered_key_le}")
if recovered_key_le != key:
    print(
        f"\033[91mERROR\033[0m: Key recovery failed.\n"
        f"  Got     : 0x{recovered_key_le}\n"
        f"  Expected: 0x{key}\n"
    )
else:
    print("\033[92mSUCCESS\033[0m: Key correctly recovered!\n")

toc = time.perf_counter()
print(f"Full key recovery phase completed in {(toc - tic)/60:.2f} minutes.")


=== Recovering k0 (most significant 64 bits) ===


Key bit recovery (k0): 100%|██████████| 34/34 [02:33<00:00,  4.52s/it]


Recovered most significant half of the key (k0): 0706050403020100

=== Recovering k1 (least significant 64 bits) ===


Key bit recovery (k1): 100%|██████████| 37/37 [02:47<00:00,  4.53s/it]

Recovered least significant half of the key (k1): 0F0E0D0C0B0A0908

Recovered full key (k1||k0): 0F0E0D0C0B0A09080706050403020100
Recovered key : 0x000102030405060708090A0B0C0D0E0F
SUCCESS: Key correctly recovered!

Full key recovery phase completed in 5.35 minutes.
